In [59]:
import pandas as pd

In [60]:
loans = pd.read_csv('data/library_loans_clean.csv', parse_dates=['loan_date', 'return_date'])
members_raw = pd.read_csv('data/library_members.csv') 
members = members_raw.copy()

print(loans.shape)
print(loans.dtypes)
print(loans.isna().sum())
print((loans.isna().mean() * 100).round(1))


(600, 8)
loan_id                   str
member_id                 str
title                     str
genre                     str
loan_date      datetime64[us]
return_date    datetime64[us]
late_fee              float64
returned                 bool
dtype: object
loan_id         0
member_id       0
title           0
genre           0
loan_date       3
return_date    36
late_fee        0
returned        0
dtype: int64
loan_id        0.0
member_id      0.0
title          0.0
genre          0.0
loan_date      0.5
return_date    6.0
late_fee       0.0
returned       0.0
dtype: float64


In [61]:
print(members.shape)
print(members.info())

(148, 3)
<class 'pandas.DataFrame'>
RangeIndex: 148 entries, 0 to 147
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   member_id   148 non-null    str    
 1   branch      136 non-null    str    
 2   birth_year  130 non-null    float64
dtypes: float64(1), str(2)
memory usage: 3.6 KB
None


In [62]:
print(loans[loans['loan_date'].isna()])

     loan_id member_id               title     genre loan_date return_date  \
462  L-30463     M0007  Blue Moon Protocol    Sci-fi       NaT  2026-05-24   
467  L-30468     M0018  Blue Moon Protocol    Sci-fi       NaT  2026-06-08   
571  L-30572     M0090  Mimmi and the Moon  Children       NaT  2026-06-26   

     late_fee  returned  
462       0.0      True  
467       0.0      True  
571       0.0      True  


In [63]:
missing_return = loans['return_date'].isna()
print((missing_return & (loans['returned'] == False)).sum())   # 30 — reason
print((missing_return & (loans['returned'] == True)).sum())    # 6  — accident

30
6


In [64]:
loans['has_return_date'] = loans['return_date'].notna()
print(loans['has_return_date'].value_counts())

has_return_date
True     564
False     36
Name: count, dtype: int64


In [65]:
print((~loans['returned'] & loans['has_return_date']).sum())

0


In [66]:
print(members['birth_year'].describe())
print((members['birth_year'] == 1900).sum())
print((members['birth_year'] == 2030).sum())


count     130.000000
mean     1983.384615
std        25.158223
min      1900.000000
25%      1978.250000
50%      1988.000000
75%      1996.750000
max      2030.000000
Name: birth_year, dtype: float64
8
3


In [67]:
print(members.duplicated().sum())
print(members.duplicated(subset='member_id').sum())

4
4


# Problems in members:
 1) 12 missing branch
 2) 18 missing birth_year
 3) 8 members with birth_year = 1900 (placeholder?)
 4) 3 members with birth_year = 2030 (error)
 5) 4 duplicate member_id

In [68]:
members['birth_year'] = members['birth_year'].where(
    members['birth_year'].between(1920, 2015)
)

print(members['birth_year'].isna().sum())
print(members['birth_year'].describe().round(0))

29
count     119.0
mean     1988.0
std        12.0
min      1957.0
25%      1979.0
50%      1988.0
75%      1996.0
max      2010.0
Name: birth_year, dtype: float64


In [69]:
print(members.dropna().shape) 
print(members.dropna(subset=['branch']).shape)
print(members.dropna(subset=['birth_year']).shape)

(107, 3)
(136, 3)
(119, 3)


In [70]:
members['branch'] = members['branch'].fillna('Unknown')
print(members['branch'].value_counts())

branch
Central    40
South      34
North      32
East       17
Harbour    13
Unknown    12
Name: count, dtype: int64


In [71]:
median = members['birth_year'].median()
filled = members['birth_year'].fillna(median)

print('median:', median)
print('before:', members['birth_year'].mean().round(1), members['birth_year'].std().round(1))
print('after :', filled.mean().round(1), filled.std().round(1))

median: 1988.0
before: 1987.8 12.1
after : 1987.9 10.8


In [77]:
print(loans.shape)
print(members.shape)
print(loans['late_fee'].sum())

merged = loans.merge(members, on='member_id', how='left')
print(merged.shape)
print(merged['late_fee'].sum())

(600, 9)
(148, 3)
7487.5
(616, 11)
7725.0


In [73]:
merged_v = loans.merge(members, on='member_id', how='left', validate='many_to_one')

MergeError: Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
 member_id
    M0015
    M0033
    M0065
    M0071 ...

In [85]:
members = members.drop_duplicates().reset_index(drop=True)
print(members.shape)

merged = loans.merge(members, on='member_id',
                     how='left',
                     validate='many_to_one',
                     indicator=True)
print(merged.shape)
print(merged['late_fee'].sum())
print(merged['_merge'].value_counts())

(144, 3)
(600, 12)
7487.5
_merge
both          569
left_only      31
right_only      0
Name: count, dtype: int64
